# Perturbed function: losses, error and machine classes

This notebook accompanies Exercise 7.1. The exact function is known, so we can distinguish empirical loss, computed on finitely many data points, from functional error. We also compare the sensitivity of squared, absolute and Huber losses to a few outliers.

## Outline

1. [Data and perturbations](#donnees-regression)
2. [One machine, three losses](#pertes-regression)
3. [Several machine classes](#machines-regression)
4. [The SVR tube](#tube-svr)
5. [Empirical loss and functional error](#erreur-fonctionnelle)

In [ ]:
import jax
import jax.numpy as jnp
import flax
from flax import nnx
import optax
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer
from sklearn.svm import SVR

jax.config.update("jax_enable_x64", True)
print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, scikit-learn {sklearn.__version__}"
)

<a id="donnees-regression"></a>
## 1. Data and perturbations

Construct data for $f_*(x)=\sin(2\pi x)$. From the same slightly perturbed observations, form a second set by shifting three values substantially. This is a constructed experiment; it does not claim to reveal an underlying probability distribution.

In [ ]:
# Complete this section.

<a id="pertes-regression"></a>
## 2. One machine, three losses

Fix sixteen piecewise affine hat functions and learn only their coefficients. The machine class is therefore identical in all three experiments; only the loss function changes.

Compare the fits obtained with

\[
\ell_2(r)=\tfrac12r^2,\qquad
\ell_1(r)=|r|,\qquad
\ell_{
m Huber}(r)=h_\delta(r).
\]

In [ ]:
noeuds = jnp.linspace(0.0, 1.0, 16)
pas_noeuds = noeuds[1] - noeuds[0]


def base_p1(x):
    x = jnp.asarray(x).reshape(-1, 1)
    return jnp.maximum(1.0 - jnp.abs(x - noeuds.reshape(1, -1)) / pas_noeuds, 0.0)


def perte_residuelle(residu, nom, delta=0.15):
    if nom == "squared":
        return 0.5 * residu**2
    if nom == "absolute":
        return jnp.abs(residu)
    if nom == "Huber":
        valeur = jnp.abs(residu)
        return jnp.where(valeur <= delta, 0.5 * residu**2, delta * (valeur - 0.5 * delta))
    raise ValueError(nom)


def ajuster_p1(x, z, nom_perte, *, lamb=1e-4, alpha=0.03, iterations=2500):
    B = base_p1(x)
    z = jnp.asarray(z)

    def objectif(p):
        residu = B @ p - z
        regularisation = 0.5 * lamb * jnp.sum(jnp.diff(p) ** 2)
        return jnp.mean(perte_residuelle(residu, nom_perte)) + regularisation

    valeur_gradient = jax.jit(jax.value_and_grad(objectif))
    transformation = optax.adam(alpha)
    p = jnp.zeros(noeuds.size)
    etat = transformation.init(p)
    for _ in range(iterations):
        valeur, gradient = valeur_gradient(p)
        mises_a_jour, etat = transformation.update(gradient, etat, p)
        p = optax.apply_updates(p, mises_a_jour)
    return p, float(valeur)

In [ ]:
# Complete this section.

<a id="machines-regression"></a>
## 3. Several machine classes

Now compare four classes: polynomials, P1 functions, Gaussian-kernel SVR and MLP. Choose hyperparameters using validation, then measure errors on a fine grid where $f_*$ is known.

In [ ]:
class MLP(nnx.Module):
    def __init__(self, largeur, *, rngs):
        self.W1 = nnx.Linear(1, largeur, rngs=rngs)
        self.W2 = nnx.Linear(largeur, largeur, rngs=rngs)
        self.W3 = nnx.Linear(largeur, 1, rngs=rngs)

    def __call__(self, x):
        x = nnx.tanh(self.W1(x))
        x = nnx.tanh(self.W2(x))
        return self.W3(x)


def norme_parametres(machine):
    return sum(jnp.sum(feuille**2) for feuille in jax.tree.leaves(nnx.state(machine, nnx.Param)))


def perte_mlp(machine, x, z, lamb=1e-5, delta=0.15):
    residu = machine(x).reshape(-1) - z
    valeur = jnp.abs(residu)
    huber = jnp.where(valeur <= delta, 0.5 * residu**2, delta * (valeur - 0.5 * delta))
    return jnp.mean(huber) + 0.5 * lamb * norme_parametres(machine)


@nnx.jit
def pas_mlp(machine, optimizer, x, z, lamb):
    valeur, gradient = nnx.value_and_grad(perte_mlp)(machine, x, z, lamb)
    optimizer.update(machine, gradient)
    return valeur


def entrainer_mlp(machine, x, z, *, lamb=1e-5, alpha=2e-3, iterations=2500):
    optimizer = nnx.Optimizer(machine, optax.adam(alpha), wrt=nnx.Param)
    x = jnp.asarray(x).reshape(-1, 1)
    z = jnp.asarray(z)
    for _ in range(iterations):
        valeur = pas_mlp(machine, optimizer, x, z, lamb)
    return float(valeur)


def predire_mlp(machine, x):
    return np.asarray(machine(jnp.asarray(x).reshape(-1, 1))).reshape(-1)

In [ ]:
# Complete this section.

<a id="tube-svr"></a>
## 4. The SVR tube

Plot the tube of half-width $\epsilon$ and the support vectors. Vary $\epsilon$: how do the number of support vectors and functional error change?

In [ ]:
# Complete this section.

<a id="erreur-fonctionnelle"></a>
## 5. Empirical loss and functional error

For each machine, compute RMSE and MAE on training and validation data, then approximate $L^2$ and $L^\infty$ errors on the grid. These quantities answer different questions.

In [ ]:
# Complete this section.

## Summary

The loss function and machine class are distinct choices. The robustness of $L^1$ or Huber concerns residual influence, while the machine class governs accessible shapes. SVR adds example selection through its tube; the MLP learns its own representations.